In [ ]:
# Cell 1: Setup and Imports
!pip install transformers torch python-docx PyPDF2 accelerate bitsandbytes -q
import torch
import os
import glob
print(f"CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'CPU'}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 13.3 MB/s eta 0:00:00
CUDA: True, Device: Tesla T4


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess
import time
import requests
import json

# Start Ollama server in background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)  # Wait for server to start

print("✅ Ollama installed and started")

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
✅ Ollama installed and started


In [ ]:
import os
import torch
import docx
import PyPDF2
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from google.colab import files

In [ ]:
from google.colab import files
from docx import Document
from PyPDF2 import PdfReader
import io
import os
import glob

class FileHandler:
    def __init__(self):
        self.prompt_text = ""
        self.transcript_text = ""

    def clean_duplicates(self):
        """Remove all duplicate files with (1), (2) etc."""
        for pattern in ["*(*).docx", "*(*).pdf", "*(*).txt"]:
            for duplicate in glob.glob(pattern):
                try:
                    os.remove(duplicate)
                    print(f"🗑️ Removed: {duplicate}")
                except:
                    pass

    def upload_prompt(self):
        print("📝 Upload PROMPT template file:")
        self.clean_duplicates()  # Clean before upload
        uploaded = files.upload()

        for name, data in uploaded.items():
            if name.endswith('.docx'):
                doc = Document(io.BytesIO(data))
                text = '\n'.join([p.text for p in doc.paragraphs if p.text.strip()])
            elif name.endswith('.pdf'):
                reader = PdfReader(io.BytesIO(data))
                text = '\n'.join([page.extract_text() for page in reader.pages])
            else:
                text = data.decode('utf-8', errors='ignore')

            self.prompt_text = text
            print(f"✅ Prompt loaded: {len(text)} chars")
            break

        self.clean_duplicates()  # Clean after upload
        print("prompt: ", self.prompt_text[:50])

    def upload_transcript(self):
        print("🎙️ Upload TRANSCRIPT file:")
        self.clean_duplicates()  # Clean before upload
        uploaded = files.upload()

        for name, data in uploaded.items():
            if name.endswith('.docx'):
                doc = Document(io.BytesIO(data))
                text = '\n'.join([p.text for p in doc.paragraphs if p.text.strip()])
            elif name.endswith('.pdf'):
                reader = PdfReader(io.BytesIO(data))
                text = '\n'.join([page.extract_text() for page in reader.pages])
            else:
                text = data.decode('utf-8', errors='ignore')

            self.transcript_text = text
            print(f"✅ Transcript loaded: {len(text)} chars")
            break

        self.clean_duplicates()  # Clean after upload
        print("transcript: ", self.transcript_text[:50])

handler = FileHandler()

In [ ]:
handler.upload_prompt()
handler.upload_transcript()

if not handler.prompt_text or not handler.transcript_text:
    print("❌ Missing files! Please upload both prompt and transcript.")
else:
    print("✅ Both files ready for processing")

📝 Upload PROMPT template file:


Saving prompt_template.docx to prompt_template.docx
✅ Prompt loaded: 6933 chars
prompt:  The following AI prompt is for summarizing intervi
🎙️ Upload TRANSCRIPT file:


Saving Michael.docx to Michael.docx
✅ Transcript loaded: 6133 chars
transcript:  So this guy's ID is 462835. He's from Ireland and 
✅ Both files ready for processing


In [ ]:
import requests
import json
import time
import subprocess
import threading

class OllamaManager:
    def __init__(self):
        self.base_url = "http://localhost:11434"
        self.models = {}
        self.start_server()

    def start_server(self):
        def run_server():
            subprocess.run(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        server_thread = threading.Thread(target=run_server, daemon=True)
        server_thread.start()
        time.sleep(10)

        try:
            response = requests.get(f"{self.base_url}/api/tags", timeout=5)
            if response.status_code == 200:
                print("🟢 Ollama server running")
        except:
            print("🔴 Server starting...")
            time.sleep(10)

    def pull_model(self, model_name):
        print(f"🔄 Pulling {model_name}...")

        try:
            response = requests.post(
                f"{self.base_url}/api/pull",
                json={"name": model_name},
                stream=True,
                timeout=600  # 10 min timeout for larger models
            )

            for line in response.iter_lines():
                if line:
                    try:
                        data = json.loads(line)
                        if "status" in data:
                            status = data["status"]
                            if "completed" in data and "total" in data:
                                percent = (data["completed"] / data["total"]) * 100
                                print(f"📥 {status}: {percent:.1f}%")
                            else:
                                print(f"📥 {status}")

                        if data.get("status") == "success":
                            print(f"✅ {model_name} ready!")
                            return True

                    except json.JSONDecodeError:
                        continue

        except Exception as e:
            print(f"❌ Failed to pull {model_name}: {e}")
            return False

        return False

    def generate(self, model_name, prompt):
        try:
            print(f"🤖 Generating with {model_name}...")

            response = requests.post(
                f"{self.base_url}/api/generate",
                json={
                    "model": model_name,
                    "prompt": prompt,
                    "stream": False,
                    "options": {
                        "temperature": 0.3,
                        "num_ctx": 8192,  # Larger context for bigger models
                        "num_predict": 2048
                    }
                },
                timeout=300  # 5 min timeout for larger models
            )

            if response.status_code == 200:
                result = response.json()
                return result.get("response", "No response")
            else:
                return f"HTTP Error: {response.status_code}"

        except Exception as e:
            return f"Generation Error: {str(e)}"

    def load_models(self):
        # Larger, higher-quality models since mistral performs best
        models_to_pull = [
            "mistral:7b-instruct",      # Best performer
            "llama3.1:8b",             # Latest Llama, large
            "llama3.2:3b",             # Good baseline
            "codellama:7b-instruct",   # Good at structured output
            "llama2:13b"               # Larger version
        ]

        print("🚀 Loading larger Ollama models...")

        for model in models_to_pull:
            success = self.pull_model(model)
            if success:
                self.models[model] = True
                print(f"✅ {model} added")
            else:
                print(f"❌ {model} failed")

        print(f"\n🎯 {len(self.models)} models ready: {list(self.models.keys())}")
        return self.models

# Initialize with larger models
print("🦙 Starting Ollama with larger models...")
ollama = OllamaManager()
loaded_models = ollama.load_models()

🦙 Starting Ollama with larger models...
🟢 Ollama server running
🚀 Loading larger Ollama models...
🔄 Pulling mistral:7b-instruct...
📥 pulling manifest
📥 pulling f5074b1221da
📥 pulling f5074b1221da
📥 pulling f5074b1221da
📥 pulling f5074b1221da
📥 pulling f5074b1221da
📥 pulling f5074b1221da
📥 pulling f5074b1221da
📥 pulling f5074b1221da
📥 pulling f5074b1221da
📥 pulling f5074b1221da: 0.3%
📥 pulling f5074b1221da: 0.9%
📥 pulling f5074b1221da: 1.5%
📥 pulling f5074b1221da: 2.1%
📥 pulling f5074b1221da: 2.8%
📥 pulling f5074b1221da: 3.4%
📥 pulling f5074b1221da: 3.7%
📥 pulling f5074b1221da: 4.2%
📥 pulling f5074b1221da: 4.6%
📥 pulling f5074b1221da: 5.2%
📥 pulling f5074b1221da: 5.8%
📥 pulling f5074b1221da: 6.5%
📥 pulling f5074b1221da: 7.0%
📥 pulling f5074b1221da: 7.7%
📥 pulling f5074b1221da: 8.1%
📥 pulling f5074b1221da: 8.6%
📥 pulling f5074b1221da: 9.1%
📥 pulling f5074b1221da: 9.5%
📥 pulling f5074b1221da: 9.9%
📥 pulling f5074b1221da: 10.2%
📥 pulling f5074b1221da: 10.5%
📥 pulling f5074b1221da: 10.7%
📥 

In [ ]:
from datetime import datetime
from IPython.display import display, Markdown

def process_ollama_display(model_name, prompt, transcript):
    """Process and display in markdown style"""

    full_prompt = f"""{prompt}

TRANSCRIPT:
{transcript}

Please categorize the above transcript according to the template provided."""

    print(f"📊 Input: {len(full_prompt)} characters")

    response = ollama.generate(model_name, full_prompt)
    return response

def display_as_markdown(model_name, result):
    """Display result as markdown in notebook"""

    markdown_content = f"""
# 🤖 {model_name.upper()}

**Time:** {datetime.now().strftime('%H:%M:%S')}
**Length:** {len(result)} characters

---

{result}

---
"""

    # Display as rendered markdown in notebook
    display(Markdown(markdown_content))

    return markdown_content

def save_simple_result(model_name, result):
    """Save with simple filename that overwrites"""
    safe_name = model_name.replace(':', '_').replace('/', '_').replace('-', '_')
    filename = f"{safe_name}.txt"

    with open(filename, 'w', encoding='utf-8') as f:
        f.write(f"Model: {model_name}\n")
        f.write(f"Generated: {datetime.now()}\n")
        f.write("="*60 + "\n\n")
        f.write(result)

    print(f"💾 Saved: {filename} (overwrites on rerun)")

# Process with all models
results = {}

if not ollama.models:
    print("❌ No models available!")
else:
    print(f"🚀 Processing with {len(ollama.models)} models...\n")

    for i, model_name in enumerate(ollama.models.keys(), 1):
        print(f"🔄 [{i}/{len(ollama.models)}] {model_name}")

        try:
            result = process_ollama_display(model_name, handler.prompt_text, handler.transcript_text)
            results[model_name] = result

            # Display as markdown in notebook
            display_as_markdown(model_name, result)

            # Save simple file (overwrites)
            save_simple_result(model_name, result)

        except Exception as e:
            error_msg = f"❌ Error: {str(e)}"
            results[model_name] = error_msg
            print(error_msg)

        print("\n" + "─"*80 + "\n")

    print(f"🎉 COMPLETED! {len(results)} models processed")

🚀 Processing with 5 models...

🔄 [1/5] mistral:7b-instruct
📊 Input: 13156 characters
🤖 Generating with mistral:7b-instruct...



# 🤖 MISTRAL:7B-INSTRUCT

**Time:** 09:02:22
**Length:** 2720 characters

---

 **Person ID: 462835**

**Concerns:**
- Physical health issues in the past, including hamstring problem and back/joint pain since 2016.
- Mental health struggles in 2017 with depression.
- History of substance use, including alcohol, cigarettes, marijuana, cocaine, and MDMA, but has quit most since 2017.

**Physical Health:**
- Hamstring problem: started (unknown), currently fine.
- Back pain and joint pain: started in 2016, improved due to Hatha Yoga.

**Mental Health:**
- Depression: started in 2017, no medication taken, improved after doing Inner Engineering and Hatha Yoga.

**Intoxicants:**
- Alcohol: last consumed twice in the last 8 years.
- Cigarettes: stopped smoking since 2017.
- Marijuana: stopped using regularly since 2017, used for approximately 6 months in Netherlands.
- Cocaine and MDMA: stopped using since 2017.
- Hard drugs: used about 25 times before 2017.

**Why Sadhanapada:**
- Seeking inner transformation and a deeper connection to life.
- Interest in becoming a full-time volunteer at the ashram.

**How 7 months off:**
- Will leave his job and charity work in good terms, with other volunteers taking care of his responsibilities during his absence.

**Expectations:**

3A. Schedule: Aware of the schedule due to previous volunteering experience.
3B. Food: Vegetarian food will not be a problem.
3C. Seva or volunteering: Looking forward to the challenge.
3D. Accommodation: Mentioned accommodation will be fine but hard sometimes.
3E. Challenges during program: Ready to face limitations and challenges in Seva. No stated reason for leaving the program.

**Work Experience or Education:**
- Studied international politics.
- Current job involves charity work, fundraising for women empowerment in Tanzania and working with mentally challenged children.

**Family Agreement:**
- Family is open-minded and wants him to be more organized.

**Finances for SP:**
- Has 250 pounds currently, plans to save 300-350 pounds per month to have around 3500 pounds by the end of 8 months.

**Legal:**
- No legal issues or liabilities.

**Programs:**
- Participated in Inner Engineering, Hatha Yoga, and Sambhavi Mahamudra Kriya.
- Volunteered for 120 days at the ashram, mainly transcribing Sadhguru's videos.

**Additional Details / Involvement Observed During the Call:**
- Appears laid back and lazy while speaking.
- Not active on social media.
- Daily routine includes waking up at 5:30, doing sadhana at 7:30, working from 8:30 to 3:30, evening sadhana, cooking, monitoring activities, and going to bed by 9:30. Weekends involve more sadhana, fun activities with friends, hiking, and volunteering on Sundays.
- Plays cards with friends.
- Interviewer's impression: not an active individual.

---


💾 Saved: mistral_7b_instruct.txt (overwrites on rerun)

────────────────────────────────────────────────────────────────────────────────

🔄 [2/5] llama3.1:8b
📊 Input: 13156 characters
🤖 Generating with llama3.1:8b...



# 🤖 LLAMA3.1:8B

**Time:** 09:03:15
**Length:** 3235 characters

---

**Person ID:** 462835

**Concerns**

* Financial constraints: The applicant has £250 set aside and plans to save an additional £300-£350 per month for the next 8 months, totaling around £3,500.
* Family agreement: The applicant's family is open-minded about his decision to attend Sadhanapada.

**Medical**

* **Physical Health**
	+ Hamstring problem: Resolved
	+ Back pain and joint pain since 2016: Improved due to Hatha Yoga in 2016
* **Mental Health**
	+ Depression in 2017: Improved after doing Inner Engineering (IE) and Hatha Yoga, progressing in six months
* **Intoxicants**
	+ Alcohol: Last consumption was eight years ago, with only two instances since then
	+ Cigarettes: Stopped smoking in 2017
	+ Marijuana, cocaine, MDMA: Stopped using these substances in 2017

**Questions Asked in Interview**

1. **Why Sadhanapada**
	* The applicant wants to experience deeper dimensions of life and inner transformation offered by Sadhanapada.
	* He is also interested in becoming a full-time volunteer at the ashram.
2. **How 7 months off**
	* The applicant plans to leave his job as a school teacher and charity fundraiser for women empowerment in Tanzania.
	* His charity work will be taken care of by other volunteers while he is away.
3. **Expectations**

3A. **Schedule**: The applicant is aware of the schedule and finds it intense, but manageable.

3B. **Food**: The applicant is a vegetarian and is comfortable with the food provided in Bhiksha Hall, eating twice a day.

3C. **Seva or Volunteering**: The applicant is excited about seva and volunteering, finding it challenging but rewarding.

3D. **Accommodation**: The applicant expects to face some challenges with shared accommodation, but is willing to adapt.

3E. **Challenges during program**: The applicant anticipates struggling with his limitations and the demands of seva and volunteering.

4. **Work Experience or Education**

* The applicant has a background in international politics and currently works as a school teacher.
* He also runs a charity for women empowerment in Tanzania.

5. **Family Agreement**
	+ The applicant's family is open-minded about his decision to attend Sadhanapada.

6. **Finances for SP**: The applicant plans to save £300-£350 per month for the next 8 months, totaling around £3,500.

7. **Legal**: No legal issues or liabilities mentioned.

8. **Programs**

* The applicant has done Inner Engineering (IE) in 2018 and Hatha Yoga program.
* He found Sambhavi Mahamudra Kriya to be a transformative experience, leading to greater sensitivity and self-awareness.

9. **Volunteering**: The applicant has volunteered at the ashram for 120 days, participating in various activities including transcribing Sadhguru's videos.

10. **Additional Details / Involvement Observed During the Call**

* The applicant is laid-back and lazy while speaking.
* He takes supplements, multivitamins, and minerals daily.
* His daily routine includes sadhana at 7:30 AM, work from 8:30 AM to 3:30 PM, and evening sadhana.

11. **Any Loans**: No loans mentioned.

12. **Lifestyle**

* The applicant's daily routine includes sadhana, work, volunteering, and relaxation.
* He enjoys hiking, dancing, live music, playing cards, and rock climbing as hobbies.

---


💾 Saved: llama3.1_8b.txt (overwrites on rerun)

────────────────────────────────────────────────────────────────────────────────

🔄 [3/5] llama3.2:3b
📊 Input: 13156 characters
🤖 Generating with llama3.2:3b...


...

💾 Saved: llama3.2_3b.txt (overwrites on rerun)

────────────────────────────────────────────────────────────────────────────────

🔄 [4/5] codellama:7b-instruct
📊 Input: 13156 characters
🤖 Generating with codellama:7b-instruct...



# 🤖 CODELLAMA:7B-INSTRUCT

**Time:** 09:04:43
**Length:** 4038 characters

---


Person ID: 462835

Concerns:

* Hamstring problem
* Back pain and joint pain since 2016
* Depression in 2017, progressed through Hata yoga and IE
* Consuming drugs and alcohol is considered stupid
* Becoming more conscious after doing Hata yoga
* No legal issues or liabilities
* Taking supplements, multivitamins, and minerals

Medical:

Physical Health:

* Name of condition: hamstring problem
* Symptoms: back pain and joint pain since 2016
* When did it start: in 2016
* Frequency: daily
* Medication taken: Hata yoga
* Current condition: better now
* How it impacts their life: improved physical health
* Impact on Isha practices: improved physical health

Mental Health:

* Name of condition: depression in 2017, progressed through Hata yoga and IE
* Symptoms: not specified
* When did it start: in 2017
* Family history: not specified
* Any self-harm tendencies: no
* Medication taken: Hata yoga and IE
* Any professional help sought: not specified
* Current condition: better now
* How it impacts their life: improved mental health
* Impact on Isha practices: improved mental health

Intoxicants:

* Type of intoxicant: alcohol
* Name: not specified
* Since when they are using it: twice in the last eight years
* Frequency of consumption: not specified
* Quantity: not specified
* Last consumption: not specified
* Reasons for stopping if they quit it: no legal issues or liabilities, taking supplements, multivitamins, and minerals

Questions Asked in Interview which needs to be summarised and categorised in the following categories:

1. Why Sadhanapada:
* Reason or motivation to apply for the program or what they expect to get out of the program
* How did they hear about it?
2. How 7 months off:
* Plans for pausing studies or job
3. Expectations:
* Program has four aspects: schedule, food, seva/volunteering, and accommodation
* Schedule: how do they feel about the program schedule and anything they shared especially any concern they have about the schedule?
* Food: what do they think about food?
* Seva/Volunteering: how do they feel about the seva or volunteering and anything they shared especially any concern they have about the seva or volunteering?
* Accommodation: how do they feel about the shared accommodation and anything they shared especially any concern they have about the shared accommodation?
4. Work Experience or Education:
* Anything they shared about work experience or education
5. Family Agreement:
* How does their family or partner feel about them coming for SP program?
6. Finances for SP:
* Any notable behaviors, tone, body language can be mentioned here.
* Any yoga or spiritual practices other than Isha programs can be mentioned here.
* Any spiritual or esoteric experiences that they shared can be mentioned here.
* Interviewer impressions: what Interviewer overall felt about the applicant?
* If anything that does not fit any of the above or below categories, list it here using original speaker text (except filler words).
7. Additional Details / Involvement Observed During the Call:
* Any notable behaviors, tone, body language can be mentioned here.
* Any yoga or spiritual practices other than Isha programs can be mentioned here.
* Any spiritual or esoteric experiences that they shared can be mentioned here.
* Interviewer impressions: what Interviewer overall felt about the applicant?
* If anything that does not fit any of the above or below categories, list it here using original speaker text (except filler words).
8. Lifestyle:
* What is their daily routine or schedule?
* What physical activities they do and diet they have?
* Any notable behaviors, tone, body language can be mentioned here.
* Any yoga or spiritual practices other than Isha programs can be mentioned here.
* Any spiritual or esoteric experiences that they shared can be mentioned here.
* Interviewer impressions: what Interviewer overall felt about the applicant?
* If anything that does not fit any of the above or below categories, list it here using original speaker text (except filler words).

---


💾 Saved: codellama_7b_instruct.txt (overwrites on rerun)

────────────────────────────────────────────────────────────────────────────────

🔄 [5/5] llama2:13b
📊 Input: 13156 characters
🤖 Generating with llama2:13b...



# 🤖 LLAMA2:13B

**Time:** 09:05:55
**Length:** 1908 characters

---


Sure! Here is the transcript categorized according to the template:

**Concerns**

* Medical: None
* Intoxicants: None

**Why Sadhanapada**

* Reason for applying: Seeks inner transformation and a deeper dimension of life.
* Motivation: Wants to be a full-time volunteer in the ashram.

**How 7 months off**

* Plans for pausing studies or job: Leaving his job and studying international politics.

**Expectations**

* Schedule: Aware of the program schedule and comfortable with it.
* Food: Knows that the food will be vegetarian and has experience with a similar diet.
* Seva or volunteering: Excited about the opportunity to serve and willing to face challenges.
* Accommodation: Aware of the shared accommodation and the potential challenges.

**Challenges during program**

* Possible reasons for leaving the program: None mentioned.

**Work Experience or Education**

* Currently works in a school and has experience with fundraising and charity work.

**Family Agreement**

* Family support: His family is open-minded and supports his decision to attend the program.

**Finances for SP**

* No loans, savings of 250 pounds, and a fund created for the program.

**Programs**

* Experience with Isha programs: Hatha Yoga, Inner Engineering, and Sambhavi Mahamudra Kriya.

**Volunteering**

* Various volunteering experiences, including transcribing Sadhguru's videos.

**Additional Details/Involvement Observed During the Call**

* Physical health: Had a hamstring problem but is fine now.
* Mental health: Was depressed and anxious in 2017 but has improved with Hata Yoga and IE.
* Intoxicants: Has stopped consuming hard drugs, marijuana, cocaine, and MDMA since 2017.
* Daily routine: Wakes up at 5:30, does sadhana at 7:30, works from 8:30 to 3:30, and spends the evening doing cooking, relaxation, and evening sadhana.

I hope this helps! Let me know if you have any further questions or requests.

---


💾 Saved: llama2_13b.txt (overwrites on rerun)

────────────────────────────────────────────────────────────────────────────────

🎉 COMPLETED! 5 models processed


In [ ]:
import re

def analyze_model_quality(results):
    """Analyze model performance with basic insights"""

    if not results:
        print("❌ No results to analyze")
        return

    print("# 📊 Model Performance Analysis\n")

    model_analysis = []

    for model_name, result in results.items():
        if "Error" in result:
            continue

        # Basic metrics
        length = len(result)
        word_count = len(result.split())
        line_count = len(result.split('\n'))

        # Content density analysis
        avg_words_per_line = word_count / max(line_count, 1)

        # Structure indicators (without keywords)
        has_sections = line_count > 10  # Multi-section response
        has_details = word_count > 200  # Detailed response
        well_formatted = avg_words_per_line < 20  # Good line breaks

        # Completeness score
        completeness = min(word_count / 100, 10)  # Cap at 10

        # Structure score
        structure_score = 0
        if has_sections: structure_score += 3
        if has_details: structure_score += 3
        if well_formatted: structure_score += 2

        # Overall quality score
        quality_score = completeness + structure_score

        model_analysis.append({
            'model': model_name,
            'words': word_count,
            'lines': line_count,
            'avg_words_per_line': round(avg_words_per_line, 1),
            'completeness': round(completeness, 1),
            'structure': structure_score,
            'quality': round(quality_score, 1)
        })

    # Sort by quality score
    model_analysis.sort(key=lambda x: x['quality'], reverse=True)

    # Display ranking table
    print("| Rank | Model | Words | Lines | Avg/Line | Quality |")
    print("|------|-------|-------|-------|----------|---------|")

    for i, stats in enumerate(model_analysis, 1):
        rank_emoji = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "📊"
        print(f"| {rank_emoji} {i} | {stats['model']} | {stats['words']} | {stats['lines']} | {stats['avg_words_per_line']} | {stats['quality']} |")

    # Insights
    if model_analysis:
        best = model_analysis[0]
        worst = model_analysis[-1]

        print(f"\n## 🎯 Key Insights")
        print(f"- **Best performer:** {best['model']} (Quality: {best['quality']})")
        print(f"- **Most detailed:** {max(model_analysis, key=lambda x: x['words'])['model']}")
        print(f"- **Best structured:** {max(model_analysis, key=lambda x: x['structure'])['model']}")
        print(f"- **Quality range:** {worst['quality']} - {best['quality']}")

        # Recommendations
        top_models = [m['model'] for m in model_analysis[:2]]
        print(f"\n## 💡 Recommendations")
        print(f"- Use **{top_models[0]}** for best overall results")
        if len(top_models) > 1:
            print(f"- **{top_models[1]}** as backup option")
        print(f"- Consider models with Quality > {sum(m['quality'] for m in model_analysis) / len(model_analysis):.1f} (average)")

# Analyze results
analyze_model_quality(results)

# 📊 Model Performance Analysis

| Rank | Model | Words | Lines | Avg/Line | Quality |
|------|-------|-------|-------|----------|---------|
| 🥇 1 | llama3.1:8b | 871 | 137 | 6.4 | 16.7 |
| 🥈 2 | llama3.2:3b | 518 | 83 | 6.2 | 13.2 |
| 🥉 3 | mistral:7b-instruct | 499 | 51 | 9.8 | 13.0 |
| 📊 4 | codellama:7b-instruct | 402 | 55 | 7.3 | 12.0 |
| 📊 5 | llama2:13b | 214 | 25 | 8.6 | 10.1 |

## 🎯 Key Insights
- **Best performer:** llama3.1:8b (Quality: 16.7)
- **Most detailed:** llama3.1:8b
- **Best structured:** llama3.1:8b
- **Quality range:** 10.1 - 16.7

## 💡 Recommendations
- Use **llama3.1:8b** for best overall results
- **llama3.2:3b** as backup option
- Consider models with Quality > 13.0 (average)
